# Percussion Track — Physical Sound Synthesis

Milestones **P0–P7** of the percussion ladder, proven in the Python lab.

## The paradigm shift (why this track is different)
Everything in the woodwind/brass/string tracks rested on three assumptions that
**all fail for percussion**:

1. **1-D  ->  2-D/3-D.** A drumhead is a *membrane*, a cymbal a *plate*, a bell a
   *shell*. d'Alembert's two counter-propagating waves (the reason a waveguide has
   two delay lines) does not survive the jump to a surface.
2. **Harmonic  ->  inharmonic.** Modal frequencies come from Bessel zeros (membrane),
   beam eigenvalues (bar), and biharmonic eigenvalues (plate) — none integer ratios.
   *Indefinite pitch is the default; definite pitch must be engineered.*
3. **Sustained  ->  struck-and-decay.** No McIntyre-Schumacher-Woodhouse feedback
   loop. A mallet deposits energy in ~1 ms; the structure rings down freely. The
   nonlinearity moves from a self-oscillating valve to a brief *contact collision*.

**Consequence:** the waveguide stops being the natural primitive. The workhorse is
**modal synthesis** — a struck linear object is a bank of damped 2nd-order
resonators, one per mode, each with a frequency, a decay, and a gain that depends
on *where you strike* (the mode shape at that point). The modes still come **from
the geometry**, so pitch emerges from physics — the project's core principle
survives, in a new representation.

**Honest boundary:** linear modal synthesis genuinely *cannot* produce a crash
cymbal or a gong bloom — those are nonlinear (mode-to-mode energy cascade) and need
FDTD or nonlinear modal coupling. That is milestone **P8**, deliberately out of
scope here and flagged at the end.

In [ ]:
import numpy as np
from scipy.signal import lfilter
from scipy.special import jn_zeros, jv
import matplotlib.pyplot as plt
from IPython.display import Audio, display

FS = 44100

## P0 — Modal resonator bank  *(the Karplus-Strong of percussion)*

A struck linear object = a sum of damped modes. Each mode is a **two-pole
resonator** with poles at `R*exp(+-j*w)`:
```
y[n] = 2 R cos(w) y[n-1]  -  R^2 y[n-2]  +  x[n]
```
Its impulse response is `R^n * sin(w n)` — a decaying sinusoid, i.e. exactly one
mode. `w = 2*pi*f/fs` sets the pitch; `R = exp(-1/(tau*fs))` sets the decay, where
`tau` is the amplitude e-folding time in seconds. This is the minimal struck
resonator, and (like Karplus-Strong for the waveguide) it is the sanity check that
the whole modal machine is wired correctly before any geometry goes in.

In [ ]:
def mode_resonator(x, f, tau, fs=FS):
    """One damped mode as a 2-pole resonator driven by excitation x."""
    w = 2*np.pi*f/fs
    R = np.exp(-1.0/(tau*fs))               # pole radius <-> decay time
    b = [(1.0 - R)]                          # keep resonant peak ~ O(1)
    a = [1.0, -2*R*np.cos(w), R*R]           # poles at R e^{+-jw}
    return lfilter(b, a, x)

def render_modes(x, freqs, taus, gains, fs=FS):
    """Sum a bank of modal resonators sharing one excitation x."""
    out = np.zeros(len(x))
    for f, tau, g in zip(freqs, taus, gains):
        if f <= 0 or f >= fs/2:              # skip DC / above Nyquist (passivity)
            continue
        out += g * mode_resonator(x, f, tau, fs)
    return out

## P1 — Contact excitation  *(the mallet)*

Two independent physical facts:

1. **Brightness = contact time.** A mallet is a short force pulse. Model it as a
   raised-cosine (Hann) bump of duration `t_c`. Its spectrum rolls off above
   `~1/t_c`, so a **hard/short** strike is broadband and excites high modes
   (bright), a **soft/long** strike is low-pass (dull). *(A full Hertzian contact
   model would make `t_c` amplitude-dependent; the parametrized pulse captures the
   audible essence — flagged as a simplification.)*
2. **Strike position = mode-shape sampling.** A mode is excited in proportion to
   its shape value at the strike point. Hit a **node** and that mode stays silent;
   hit an **antinode** and it rings maximally. This is why a *center* drum strike
   sounds unpitched and an *off-center* strike sings — same logic for every family.

In [ ]:
def contact_pulse(n_samples, t_c, fs=FS):
    """Raised-cosine force bump; t_c (s) sets brightness (short=hard=bright)."""
    L = max(2, int(t_c*fs))
    win = 0.5*(1 - np.cos(2*np.pi*np.arange(L)/(L-1)))
    x = np.zeros(n_samples); x[:L] = win
    return x

## Analysis helpers  *(validation gate — impulse test + spectrum, per §10)*

In [ ]:
def peaks(sig, fs=FS, n_peaks=8, fmin=20):
    """Pick the strongest spectral peaks — for comparing against predicted modes."""
    X = np.abs(np.fft.rfft(sig*np.hanning(len(sig))))
    fr = np.fft.rfftfreq(len(sig), 1/fs)
    X[fr < fmin] = 0
    idx = [i for i in range(1, len(X)-1)
           if X[i] > X[i-1] and X[i] >= X[i+1] and X[i] > 0.02*X.max()]
    idx = sorted(idx, key=lambda i: -X[i])[:n_peaks]
    return sorted(fr[i] for i in idx)

def plot_spectrum(sig, title, fmax=6000, fs=FS):
    X = 20*np.log10(np.abs(np.fft.rfft(sig*np.hanning(len(sig)))) + 1e-6)
    fr = np.fft.rfftfreq(len(sig), 1/fs)
    m = fr <= fmax
    plt.figure(figsize=(9,2.6)); plt.plot(fr[m], X[m], lw=0.8)
    plt.title(title); plt.xlabel("Hz"); plt.ylabel("dB"); plt.tight_layout(); plt.show()

def norm(sig):
    return sig/(np.max(np.abs(sig)) + 1e-12)

---
# SKIN — Membranes  (2-D wave equation, Bessel modes)

A circular head of tension `T` and surface density `sigma` obeys
`d2u/dt2 = c^2 * laplacian(u)`, `c = sqrt(T/sigma)`. Clamped at the rim, the modes
are Bessel functions with
```
f_mn = (c / 2*pi*a) * j_mn        j_mn = n-th zero of J_m
```
The zeros give a stubbornly **inharmonic** ratio set (1.00, 1.59, 2.14, 2.30, ...),
which is *why an ideal drumhead has no pitch*. Mode shape is
`J_m(j_mn * r/a) * cos(m*theta)`; since `J_m(0)=0` for `m>0`, a **center strike**
excites only the axisymmetric `(0,n)` breathing modes -> a dull, pitchless thud.

## P2 — Ideal circular membrane

In [ ]:
def membrane_modes(f01=110.0, n_m=6, n_n=4):
    """(m, freqs) for an ideal circular membrane, scaled so mode (0,1)=f01."""
    j01 = jn_zeros(0, 1)[0]
    mm, ff = [], []
    for m in range(n_m):
        for z in jn_zeros(m, n_n):
            mm.append(m); ff.append(f01 * z / j01)
    return np.array(mm), np.array(ff)

def _membrane_gain(m_arr, f_arr, f01, rho):
    """Radial mode-shape amplitude at r/a = rho (theta=0). rho=0 => center strike."""
    j01 = jn_zeros(0,1)[0]
    return np.array([abs(jv(m, (f/f01*j01)*rho)) for m, f in zip(m_arr, f_arr)])

def drum(f01=110.0, rho=0.55, t_c=0.0012, dur=1.2, tau0=0.35, radiate_hi=0.6):
    """Ideal membrane. rho=0 center (unpitched); rho~0.5-0.7 = musical strike."""
    n = int(dur*FS); x = contact_pulse(n, t_c)
    m_arr, f_arr = membrane_modes(f01)
    gains = _membrane_gain(m_arr, f_arr, f01, rho) * (f_arr/f01)**(-radiate_hi)
    taus  = tau0 * (f01/np.maximum(f_arr, f01))**0.7   # high modes decay faster
    return render_modes(x, f_arr, taus, gains), f_arr

In [ ]:
sig_center,_  = drum(rho=0.0)
sig_offctr,pf = drum(rho=0.55)
print("center RMS:", round(np.sqrt(np.mean(sig_center**2)),4),
      " off-center RMS:", round(np.sqrt(np.mean(sig_offctr**2)),4))
print("predicted first modes:", [round(f,1) for f in pf[:6]])
print("measured  (off-center):", [round(f,1) for f in peaks(sig_offctr, n_peaks=6)])
plot_spectrum(sig_offctr, "P2 ideal membrane — inharmonic, no pitch")
display(Audio(norm(sig_offctr), rate=FS))

## P3 — Kettle-coupled membrane  ->  **timpani**  *(pitch from nowhere)*

The surprise of the whole track. Air loading (the kettle) plus the membrane's own
radiation mass **pull the ring modes** `(1,1),(2,1),(3,1),...` into a *near-harmonic*
`2:3:4:5:6` series over a **missing fundamental**, while the poorly-radiating
axisymmetric `(0,n)` modes are **heavily damped**. Definite pitch *emerges from the
coupling* — and only off-center, because a center strike wakes the unpitched `(0,1)`.

**Honest limitation:** we do not solve the coupled membrane-air PDE here — we apply
its *measured result* (the air-loaded ratios, after Rossing) to the modal bank, so
the spectrum shows what the coupling does. A first-principles version would solve
the fluid-loaded eigenproblem; that is a heavier future step.

In [ ]:
TIMPANI_RATIOS = [1.00, 1.50, 1.99, 2.44, 2.88, 3.32]   # (1,1)..(6,1), air-loaded

def timpani(f_pitch=147.0, t_c=0.0018, dur=2.2, tau0=1.4):
    n = int(dur*FS); x = contact_pulse(n, t_c)
    freqs, taus, gains = [], [], []
    for k, r in enumerate(TIMPANI_RATIOS):              # tuned, ringing ring-modes
        freqs.append(f_pitch*r); taus.append(tau0*(1.0/r)**0.5)
        gains.append(1.0/(1+k*0.35))
    for r in [1.52, 2.30]:                              # axisymmetric "thud", damped
        freqs.append(f_pitch*r*1.03); taus.append(0.18); gains.append(0.25)
    return render_modes(x, np.array(freqs), np.array(taus), np.array(gains))

In [ ]:
sig = timpani(147.0)
print("predicted ring modes:", [round(147*r,1) for r in TIMPANI_RATIOS])
print("measured            :", [round(f,1) for f in peaks(sig, n_peaks=7)])
plot_spectrum(sig, "P3 timpani — ring modes pulled to near-harmonic (definite pitch)")
display(Audio(norm(sig), rate=FS))

---
# BLOCK — Stiff bars  (Euler-Bernoulli beam, 4th order)

A bar is restored by **bending stiffness**, not tension, so it is 4th-order in
space: `d2u/dt2 = -(EI/rho A) * d4u/dx4`. Free-free eigenfrequencies scale as
`(beta_n L)^2`, giving the ratios `1 : 2.756 : 5.40 : 8.93` — even more inharmonic
than a membrane. That is the raw "clunk" of a woodblock or clave. Cross-track reuse:
this stiffness dispersion is the *same* physics as piano-string inharmonicity in
the string ladder.

## P4 — Free-free stiff bar  (woodblock / clave / untuned bar)

In [ ]:
BETA_L = np.array([4.73004074, 7.85320462, 10.9956078, 14.1371655, 17.2787597])

def bar_ratios(n_modes=5):
    b = BETA_L[:n_modes]; return (b/b[0])**2

def _bar_shape(beta_l, xi):
    """Free-free beam eigenfunction at position xi in [0,1] (for strike weighting)."""
    sig = (np.cosh(beta_l)-np.cos(beta_l))/(np.sinh(beta_l)-np.sin(beta_l))
    xb = xi*beta_l
    return (np.cosh(xb)+np.cos(xb)) - sig*(np.sinh(xb)+np.sin(xb))

def bar(f1=440.0, xi=0.28, t_c=0.0008, dur=0.8, tau0=0.5, n_modes=5,
        ratios=None, gains=None):
    """Free-free bar. xi=0.5 is a NODE of every antisymmetric mode -> strike off-centre."""
    n = int(dur*FS); x = contact_pulse(n, t_c)
    r = bar_ratios(n_modes) if ratios is None else np.array(ratios)
    freqs = f1*r
    taus  = tau0*(1.0/r)**0.6
    if gains is None:
        g = np.array([abs(_bar_shape(BETA_L[i], xi)) for i in range(len(r))]) / (r**0.3)
    else:
        g = np.array(gains)
    return render_modes(x, freqs, taus, g), freqs

In [ ]:
sig, freqs = bar(440.0, xi=0.28)
print("bar ratios (want 1, 2.756, 5.40, 8.93):", np.round(bar_ratios(),3))
print("predicted:", [round(f,0) for f in freqs])
# high modes decay fast; look in the 25 ms attack where they still live
print("attack-window peaks:", [round(f,0) for f in peaks(sig[:int(0.025*FS)], n_peaks=4, fmin=300)])
plot_spectrum(sig, "P4 free-free bar — strongly inharmonic 'clunk'")
display(Audio(norm(sig), rate=FS))

## P5 — Undercut bar + tuned tube  ->  **marimba / xylophone**

**Undercutting** (arching out the underside) pins the first overtone to `3:1`
(xylophone, a twelfth) or `4:1` (marimba, a double octave); a **tuned tube**
resonator underneath reinforces the fundamental (the same Helmholtz resonance work
as brass B3).

**Honest limitation:** undercutting *reshapes the mode shapes* — the tuned overtone
is no longer the ideal antisymmetric free-free mode with a node at centre. So here
we (a) impose the tuned *frequencies* (the audible signature) and (b) supply
**explicit modal gains** rather than the now-invalid ideal eigenfunctions, and
strike near centre as a player does. A first-principles version would solve the
variable-cross-section eigenproblem for both frequencies *and* shapes.

In [ ]:
def marimba(f1=220.0, kind="marimba", t_c=0.0011, dur=1.4, tau0=0.9):
    over = 4.0 if kind == "marimba" else 3.0          # marimba 4:1, xylophone 3:1
    ratios = [1.0, over, 9.2, 16.7]
    gains  = [1.0, 0.55, 0.18, 0.08]                  # explicit: undercut reshapes modes
    sig, freqs = bar(f1, t_c=t_c, dur=dur, tau0=tau0,
                     n_modes=4, ratios=ratios, gains=gains)
    n = int(dur*FS); x = contact_pulse(n, t_c)
    tube = render_modes(x, [f1], [tau0*1.3], [1.4])   # tuned tube reinforces f1
    return sig + tube, freqs

In [ ]:
for kind, f1 in [("marimba",220.0), ("xylophone",440.0)]:
    sig, freqs = marimba(f1, kind)
    print(f"{kind:9s} predicted:", [round(f,0) for f in freqs],
          " measured:", [round(f,0) for f in peaks(sig, n_peaks=3)])
    display(Audio(norm(sig), rate=FS))
plot_spectrum(marimba(220.0,"marimba")[0], "P5 marimba — fundamental + tuned 4:1 overtone")

---
# BELL — Plates & shells  (biharmonic / shell equations)

A flat plate obeys `d2u/dt2 = -(D/rho h) * biharmonic(u)` — a dense, inharmonic
modal thicket (struck plate, splash). A **bell** is a curved *shell*: the curvature
couples modes and profile-shaping tunes named partials. The perceived **strike
note** is a *virtual pitch* inferred from the upper partials — a literal
missing-fundamental phenomenon (ties straight to §3.1).

## P6 — Flat plate (linear)  (struck plate / splash)

Using the simply-supported rectangular plate, whose eigenvalues are closed-form and
inharmonic: `f_mn ~ (m^2/Lx^2 + n^2/Ly^2)`, shape `sin(m*pi*x/Lx) sin(n*pi*y/Ly)`.

In [ ]:
def plate(f11=180.0, ar=1.31, nx=5, ny=5, t_c=0.0006, dur=1.0, tau0=0.6,
          px=0.42, py=0.37):
    n = int(dur*FS); x = contact_pulse(n, t_c)
    base = f11/(1 + 1/ar**2)                          # so mode (1,1) lands on f11
    freqs, taus, gains = [], [], []
    for m in range(1, nx+1):
        for k in range(1, ny+1):
            f = base*(m**2 + (k/ar)**2)
            freqs.append(f); taus.append(tau0*(f11/max(f,f11))**0.8)
            gains.append(abs(np.sin(m*np.pi*px)*np.sin(k*np.pi*py)))
    freqs = np.array(freqs)
    sig = render_modes(x, freqs, np.array(taus), np.array(gains))
    return sig, np.sort(freqs)[:8]

In [ ]:
sig, pf = plate(180.0)
print("predicted (first 8):", [round(f,0) for f in pf])
print("measured           :", [round(f,0) for f in peaks(sig, n_peaks=8)])
plot_spectrum(sig, "P6 struck plate — dense, metallic, inharmonic")
display(Audio(norm(sig), rate=FS))

## P7 — Shell  ->  **tuned church bell**  (named partials, strike note)

Profile-tuned partials over the **prime**. The **tierce** is a *minor third*
(`6/5 = 1.2`) — the melancholy signature of Western bells. The **strike note** is a
virtual pitch heard *at the prime*, an octave below the nominal, reconstructed by
the ear from the upper partials (missing fundamental, §3.1). Bells ring **long**;
the hum longest.

In [ ]:
BELL = [("hum",0.5,4.0,1.0), ("prime",1.0,3.0,1.0), ("tierce",1.2,2.2,0.9),
        ("quint",1.5,1.6,0.7), ("nominal",2.0,1.3,0.9),
        ("sup-quint",3.0,0.8,0.5), ("oct-nominal",4.0,0.6,0.4)]

def bell(f_prime=440.0, t_c=0.0007, dur=4.0):
    n = int(dur*FS); x = contact_pulse(n, t_c)
    freqs = [f_prime*r for _,r,_,_ in BELL]
    taus  = [t for *_,t,_ in BELL]
    gains = [g for *_,g in BELL]
    return render_modes(x, np.array(freqs), np.array(taus), np.array(gains)), np.array(freqs)

In [ ]:
sig, pf = bell(440.0)
for (name,r,_,_), f in zip(BELL, pf):
    print(f"  {name:11s} {r:>4}x  ->  {f:6.0f} Hz")
print("measured:", [round(f,0) for f in peaks(sig, n_peaks=7)])
plot_spectrum(sig, "P7 tuned bell — hum/prime/tierce(minor-3rd)/quint/nominal")
display(Audio(norm(sig), rate=FS))

---
# P8 — Nonlinear plate  ->  cymbal / gong  *(the bowed-string of this track)*

The linear modal bank (P0-P7) **cannot** make a crash: linear modes are
independent — energy struck into a mode stays there and decays. A cymbal does the
opposite. Strike it and, over tens of milliseconds, energy **cascades** from the
few low modes you excited into a dense high-frequency shimmer; a tam-tam's sound
even *builds after the strike* (the "bloom"). That is **geometric nonlinearity**:
when a thin plate deflects by more than ~its own thickness, stretching of the
mid-plane couples to bending, and modes start trading energy.

The governing physics is the **Foppl-von Karman** system — the displacement `w`
coupled to an in-plane (Airy) stress function `F`:
```
rho*h * w_tt = -D grad^4 w  +  L(w, F)  -  damping  +  force
     grad^4 F = -(1/2) E h  L(w, w)
L(f,g) = f_xx g_yy + f_yy g_xx - 2 f_xy g_xy      (von Karman bracket)
```
`L(w,F)` is the transverse force from in-plane stress; `L(w,w)` sources that stress
from curvature. Eliminating `F` leaves a **cubic** coupling in `w` — the mechanism
that moves energy between modes. This is no longer a bank of biquads; it is a PDE.

**Method (pseudo-spectral, simply-supported rectangular plate).** The sine basis
diagonalises `grad^4` exactly, so the stiff linear part and the `grad^4 F = ...`
inversion are two cheap transforms; the nonlinear bracket is computed by finite
differences on the grid. We keep only modes below an audio cutoff (a physical
band-limit that also guarantees stability), march with velocity-Verlet, and impose
per-mode damping. Crucially these are **the same P6 plate modes** — now *coupled*.

In [ ]:
from scipy.fft import dstn, idstn
class NLPlate:
    def __init__(self, N=44, L=0.35, E=1e11, rho=8700., nu=0.3, h=8e-4,
                 fcut=10000., fs=44100, oversample=2):
        self.N, self.L, self.h = N, L, h
        self.E, self.rho = E, rho
        self.D = E*h**3/(12*(1-nu**2))
        self.rho_h = rho*h
        self.S = np.sqrt(self.D/self.rho_h)          # wave-speed-like factor
        self.dx = L/(N+1)
        self.fs, self.os = fs, oversample
        self.dt = 1.0/(fs*oversample)

        m = np.arange(1, N+1)
        km = m*np.pi/L
        KX, KY = np.meshgrid(km, km, indexing='ij')
        self.lam  = KX**2 + KY**2                      # -Laplacian eigenvalue
        self.lam2 = self.lam**2                        # biharmonic eigenvalue
        self.fmn  = self.S*self.lam/(2*np.pi)          # linear modal frequency
        self.mask = (self.fmn <= fcut).astype(float)   # keep only audible modes
        # modal damping: higher modes decay faster (shapes shimmer fade)
        self.sig  = 1.5 + 4.0*(self.fmn/1000.0)        # 1/s per mode
        self.dmp  = np.exp(-self.sig*self.dt)*self.mask

    def _d2(self, f):
        fp = np.pad(f, 1)                              # zero (Dirichlet) boundaries
        dx2 = self.dx**2
        fxx = (fp[2:,1:-1]-2*f+fp[:-2,1:-1])/dx2
        fyy = (fp[1:-1,2:]-2*f+fp[1:-1,:-2])/dx2
        fxy = (fp[2:,2:]-fp[2:,:-2]-fp[:-2,2:]+fp[:-2,:-2])/(4*dx2)
        return fxx, fyy, fxy

    def _bracket(self, f, g):
        fxx,fyy,fxy = self._d2(f); gxx,gyy,gxy = self._d2(g)
        return fxx*gyy + fyy*gxx - 2*fxy*gxy

    def _dst2(self, x):  return dstn(x, type=1, norm='ortho')
    def _idst2(self, X): return idstn(X, type=1, norm='ortho')
    def _project(self, x): return self._idst2(self.mask*self._dst2(x))
    def _biharm(self, w):  return self._idst2(self.lam2*self._dst2(w))

    def _stress(self, w):                              # solve grad^4 F = -1/2 Eh L(w,w)
        rhs = -0.5*self.E*self.h*self._bracket(w, w)
        return self._idst2(self.mask*self._dst2(rhs)/self.lam2)

    def _accel(self, w, nonlinear=True):
        a = -(self.D/self.rho_h)*self._biharm(w)      # in-band already (w is projected)
        if nonlinear:
            F = self._stress(w)
            a = a + self._project((1.0/self.rho_h)*self._bracket(w, F))
        return a

    def strike(self, amp, dur=1.2, pos=(0.42,0.55), width=0.12,
               pickup=(0.63,0.37), nonlinear=True):
        N, L = self.N, self.L
        xs = (np.arange(1,N+1)/(N+1))
        X,Y = np.meshgrid(xs, xs, indexing='ij')
        bump = np.exp(-((X-pos[0])**2+(Y-pos[1])**2)/(2*width**2))
        w = np.zeros((N,N)); v = self._project(amp*bump)   # struck: initial velocity
        px = int(round(pickup[0]*(N+1)))-1; py = int(round(pickup[1]*(N+1)))-1
        a = self._accel(w, nonlinear)
        n_out = int(dur*self.fs); out = np.empty(n_out)
        dt, os = self.dt, self.os; wmax = 0.0
        for i in range(n_out):
            for _ in range(os):
                w = self._project(w + dt*v + 0.5*dt*dt*a)
                a2 = self._accel(w, nonlinear)
                v  = self._idst2(self.dmp*self._dst2(v + 0.5*dt*(a+a2)))
                a  = a2
            out[i] = w[px,py]
            wmax = max(wmax, np.abs(w).max())
        return out, wmax

    def linear_mode_freqs(self, k=12):
        f = np.sort(self.fmn[self.mask>0].ravel())
        return f[:k]

### Honesty gate: at *small* amplitude the nonlinear solver must reduce to P6
If the deflection is tiny (`max|w| << h`), the cubic term is negligible and the
pickup spectrum must land on the **linear** simply-supported plate modes
`f_mn = sqrt(D/rho*h) * (k_m^2 + k_n^2) / 2*pi`. This is the P6 validation, re-run
through the full nonlinear engine — if it fails, nothing above it can be trusted.

In [ ]:
def _pk(sig, fs=44100, n=6, fmin=15):
    X = np.abs(np.fft.rfft(sig*np.hanning(len(sig)))); fr = np.fft.rfftfreq(len(sig),1/fs)
    X[fr<fmin]=0
    idx=[i for i in range(1,len(X)-1) if X[i]>X[i-1] and X[i]>=X[i+1] and X[i]>0.05*X.max()]
    return np.round(sorted(fr[i] for i in sorted(idx,key=lambda i:-X[i])[:n]),0)

plate_nl = NLPlate(N=40, fcut=8500, oversample=1)   # ~30 s per 0.6 s of audio
soft, wmax_s = plate_nl.strike(amp=0.002, dur=0.6, nonlinear=True)
print("max|w|/h = %.4f   (<< 1  =>  linear regime)" % (wmax_s/plate_nl.h))
print("predicted linear modes:", np.round(plate_nl.linear_mode_freqs(6),0))
print("measured pickup peaks :", _pk(soft))
plot_spectrum(soft, "P8 small-amplitude strike — reduces to linear plate modes (== P6)")
display(Audio(norm(soft), rate=FS))

### The cascade: a *hard* strike generates highs that were never struck
We now strike with a **broad, strong** bump — broad so it deposits energy *only in
low modes*, strong so `max|w| ~ several*h`. If the plate were linear the high band
would stay empty. Instead the von Karman coupling fills it: the spectrogram shows
energy **climbing** out of the low modes into a broadband wash, and the 2-8 kHz
shimmer ends up orders of magnitude louder than in the soft strike. That emergent
HF wash *is* the crash; its delayed build is the gong/tam-tam bloom.

In [ ]:
def band_energy(sig, lo, hi, fs=44100):
    X=np.abs(np.fft.rfft(sig)); fr=np.fft.rfftfreq(len(sig),1/fs)
    return np.sum(X[(fr>=lo)&(fr<hi)]**2)

hard, wmax_h = plate_nl.strike(amp=6.0, dur=1.0, width=0.16, nonlinear=True)
print("HARD strike: max|w|/h = %.1f   (large-deflection, nonlinear, stable)" % (wmax_h/plate_nl.h))
e_soft = band_energy(soft, 2000, 8000)
e_hard = band_energy(hard, 2000, 8000)
print("2-8 kHz energy  soft: %.2e   hard: %.2e   ratio: %.0fx"
      % (e_soft, e_hard, e_hard/(e_soft+1e-30)))
print("=> the highs in the hard strike were GENERATED by the cascade, not struck in.")

plt.figure(figsize=(9,3))
plt.specgram(hard, NFFT=2048, Fs=FS, noverlap=1536, cmap="magma")
plt.ylim(0,8000); plt.title("P8 hard strike — energy cascades UP over time (the crash/bloom)")
plt.xlabel("s"); plt.ylabel("Hz"); plt.tight_layout(); plt.show()
display(Audio(norm(hard), rate=FS))

### Honest limitations (what this is, and what it is not)
- **Shape.** A simply-supported flat rectangular plate stands in for a real cymbal
  (circular, curved, with a cup/bell, often spinning) or gong (dished, with a
  turned rim). The *mechanism* — von Karman cubic mode-coupling — is
  geometry-agnostic and is what produces the cascade; the specific timbre of a
  named cymbal is not the claim. Real geometry needs the curved-shell operator and
  the actual boundary/profile.
- **In-plane boundary condition.** `F` is expanded in the same sine basis as `w`,
  an idealised in-plane condition (roughly movable edges); the true condition
  depends on how the plate is mounted and shifts the coupling strengths.
- **Numerics.** Explicit velocity-Verlet with a spectral band-limit; it is stable
  in-band but **not** strictly energy-conserving (Bilbao's energy-balanced schemes
  are the rigorous route). Damping is a simple frequency-rising modal model, not
  radiation/thermoelastic loss from first principles.
- **Cost.** Pure-NumPy at audio rate (~30 s per 0.6 s here). The real-time port
  would move this to the Rust/WASM tier, likely with a modal cubic-coupling tensor
  precomputed offline rather than a live pseudo-spectral solve.

What it *does* honestly demonstrate: strike a linear plate softly and you get P6's
inharmonic ping; strike the **same** plate hard and the geometric nonlinearity
alone turns it into a crash — the qualitative behaviour no linear model can produce.